<div style="background:#AA4B37;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#1C3257;font-size:12px;letter-spacing:2px;font-weight:bold;background:#F7F3EB;display:inline-block;padding:2px 8px;border-radius:4px">CLAVE DEL PROFESOR — NO DISTRIBUIR</div><div style="font-size:26px;font-weight:bold;margin-top:8px">Lab 1 — Pipeline de preprocesamiento · RESUELTO</div><div style="font-style:italic;color:#F2DDD6;margin-top:8px">Clave del profesor · NLTK + spaCy (es_core_news_sm)</div></div>

> **Notas de calificación.** Las celdas resueltas representan *una* solución correcta de referencia, no la única. 
Lo que se evalúa en la defensa oral es que el alumno pueda **justificar** sus decisiones, no que coincida 
literalmente con esta clave. Cada bloque incluye, en *Respuesta modelo*, lo mínimo que una respuesta sólida debe contener.


## 0 · Entorno

In [1]:
! pip install nltk spacy pandas 

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.15-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.3 kB)
  Using cached cymem-2.0.13-cp312-cp312-macosx_11_0_arm64.whl.metadata (9.7 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached blis-1.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (7.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 2.6 MB/s  0:00:02m 2.5 MB/s eta 0:00:01
Using cached catalogue-2.0.10-py3-none-any.whl (17 kB)
Using cached cymem-2.0.13-cp312-cp312-macosx_11_0_arm64.whl (42 kB)
Using cached murmurhash-1.0.15-cp312-cp312-macosx_11_0_arm64.whl (27 kB)
Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl (29 kB)
Using cached spacy_loggers-1.0.5-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 658.5/658.5 kB 5.7 MB/s  0:00:0

In [ ]:
# !pip install -q nltk spacy && python -m spacy download es_core_news_sm
import re, unicodedata, json
from collections import Counter
import pandas as pd
import nltk
nltk.download('punkt', quiet=True)
from nltk.stem import SnowballStemmer
import spacy
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download; download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')
print('Entorno listo.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 9.0 MB/s  0:00:018.8 MB/s eta 0:00:0101
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Entorno listo.


In [3]:
corpus_crudo = [
    {"id": "d01", "titulo": "Lluvias provocan inundaciones en Tuxtla",
     "texto": "Las  fuertes lluvias   provocaron inundaciones en varias colonias del sur de Tuxtla Gutierrez 😟. "
              "Proteccion Civil pidio a la poblacion no cruzar las calles anegadas. Mas info en https://chiapasparalelo.com/nota1 ."},
    {"id": "d02", "titulo": "Crisis hidrica golpea la region",
     "texto": "La crisis hidrica se agrava: el desabasto del liquido vital afecta a miles de familias en la zona alta. "
              "Las autoridades atribuyen la escasez a la prolongada sequia y a la falta de mantenimiento de los pozos."},
    {"id": "d03", "titulo": "Cafe de Chiapas rompe record de exportacion",
     "texto": "<p>El <b>cafe</b> de Chiapas rompio su record historico de exportacion este ciclo, impulsado por la demanda en Europa y Asia.</p> "
              "Los productores de la Sierra celebran precios al alza."},
    {"id": "d04", "titulo": "Sequia afecta cultivos de maiz",
     "texto": "La sequia afecta gravemente los cultivos de maiz y frijol en la region fronteriza. "
              "Los agricultores reportan perdidas de hasta el 40% y piden apoyos al gobierno estatal."},
    {"id": "d05", "titulo": "Turismo crece en el Canon del Sumidero",
     "texto": "El Canon del Sumidero recibio mas de 200 mil visitantes durante la temporada. "
              "Los recorridos en lancha y el avistamiento de fauna son los principales atractivos del parque nacional. #Chiapas"},
    {"id": "d06", "titulo": "Sismo de magnitud 5.1 frente a las costas",
     "texto": "Un sismo de magnitud 5.1 se registro frente a las costas de Chiapas la madrugada del martes. "
              "No se reportaron danos ni victimas, informo el Servicio Sismologico Nacional."},
    {"id": "d07", "titulo": "UPCh inaugura laboratorio de IA",
     "texto": "La Universidad Politecnica de Chiapas inauguro un nuevo laboratorio de inteligencia artificial "
              "equipado con GPUs para proyectos de aprendizaje automatico y vision por computadora. Visita https://upchiapas.edu.mx ."},
    {"id": "d08", "titulo": "Repunta la produccion de cacao",
     "texto": "La produccion de cacao en la region del Soconusco repunto este año tras varios ciclos a la baja. "
              "Cooperativas locales apuestan por el chocolate artesanal de origen para mercados premium."},
    {"id": "d09", "titulo": "San Cristobal, destino cultural",
     "texto": "San Cristobal de las Casas se consolida como destino cultural: sus mercados, iglesias y cafeterias "
              "atraen a viajeros de todo el mundo. La gastronomia y el textil artesanal son protagonistas."},
    {"id": "d10", "titulo": "Avanza obra de infraestructura carretera",
     "texto": "Avanza la rehabilitacion de la carretera que conecta Tuxtla con la costa. "
              "La obra busca reducir tiempos de traslado y mejorar la seguridad vial para miles de automovilistas."},
    {"id": "d11", "titulo": "Alertan por casos de dengue",
     "texto": "La Secretaria de Salud alerto por un repunte de casos de dengue en municipios de tierra caliente. "
              "Pide a la poblacion eliminar criaderos de mosco y acudir al medico ante fiebre alta. 🦟"},
    {"id": "d12", "titulo": "Feria celebra el cafe y el cacao",
     "texto": "La feria regional celebro el cafe y el cacao chiapaneco con catas, musica y venta directa de productores. "
              "Miles de asistentes recorrieron los stands durante el fin de semana."},
    {"id": "d13", "titulo": "Restablecen servicio de agua potable",
     "texto": "El servicio de agua potable se restablecera de forma escalonada en las colonias afectadas por la falla en la red. "
              "El organismo operador pidio a los usuarios almacenar agua de manera responsable."},
    {"id": "d14", "titulo": "Estudiantes ganan concurso de robotica",
     "texto": "Estudiantes de ingenieria de Tuxtla ganaron el primer lugar en un concurso nacional de robotica 🤖 "
              "con un brazo robotico de bajo costo. El equipo representara a Mexico en una competencia internacional."},
]
print(f"{len(corpus_crudo)} documentos cargados.")

14 documentos cargados.


In [4]:
df = pd.DataFrame(corpus_crudo)
df['n_chars'] = df['texto'].str.len()
df[['id', 'titulo', 'n_chars']]

,id,titulo,n_chars
0,d01,Lluvias provocan inundaciones en Tuxtla,213
1,d02,Crisis hidrica golpea la region,207
2,d03,Cafe de Chiapas rompe record de exportacion,184
3,d04,Sequia afecta cultivos de maiz,169
4,d05,Turismo crece en el Canon del Sumidero,190
5,d06,Sismo de magnitud 5.1 frente a las costas,170
6,d07,UPCh inaugura laboratorio de IA,213
7,d08,Repunta la produccion de cacao,186
8,d09,"San Cristobal, destino cultural",190
9,d10,Avanza obra de infraestructura carretera,173


## 1 · Cargar y explorar

**1.a** Estadísticas de longitud.

In [5]:
n_docs = len(corpus_crudo)
chars = df['n_chars']
palabras = df['texto'].str.split().str.len()
print(f'Documentos        : {n_docs}')
print(f'Long. media (char): {chars.mean():.1f}  (min {chars.min()}, max {chars.max()})')
print(f'Long. media (palabras, split ingenuo): {palabras.mean():.1f}')

Documentos        : 14
Long. media (char): 189.1  (min 169, max 213)
Long. media (palabras, split ingenuo): 30.2


**1.b** Detección de ruido.

In [6]:
RE_URL   = re.compile(r'https?://\S+')
RE_HTML  = re.compile(r'</?[a-zA-Z][^>]*>')
# rango de emojis (simbolos, pictogramas, transporte, banderas)
RE_EMOJI = re.compile('[' '\U0001F300-\U0001FAFF' '\U00002600-\U000027BF' ']+')

for fila in corpus_crudo:
    t = fila['texto']
    hallazgos = {'URL': RE_URL.findall(t), 'HTML': RE_HTML.findall(t), 'EMOJI': RE_EMOJI.findall(t)}
    hallazgos = {k: v for k, v in hallazgos.items() if v}
    if hallazgos:
        print(fila['id'], '->', hallazgos)

d01 -> {'URL': ['https://chiapasparalelo.com/nota1'], 'EMOJI': ['😟']}
d03 -> {'HTML': ['<p>', '<b>', '</b>', '</p>']}
d07 -> {'URL': ['https://upchiapas.edu.mx']}
d11 -> {'EMOJI': ['🦟']}
d14 -> {'EMOJI': ['🤖']}


**Respuesta modelo (1.b).** Los tres tipos *pueden* ser señal según la tarea: los **emojis** codifican afecto (😟 negativo, 🤖 temático) y son útiles para *sentiment analysis*; los **hashtags** (#Chiapas) son etiquetas temáticas explícitas; las **URLs** identifican la fuente y su dominio puede informar credibilidad. Para *este* buscador, sin embargo, los tres son ruido y se eliminan. Una respuesta sólida del alumno debe nombrar **al menos un caso** donde el ruido sería señal y atarlo a una **tarea concreta** distinta de la búsqueda.


## 2 · Tokenizar y normalizar

**2.a** `split` ingenuo vs. tokenizador real.

In [7]:
ejemplo = corpus_crudo[0]['texto']
ingenuo = ejemplo.split()
real = [t.text for t in nlp(ejemplo)]
print('Ingenuo:', ingenuo[:14])
print('spaCy  :', real[:18])
# Diferencias clave:
#  1) El split deja la puntuacion pegada: 'Gutierrez 😟.' -> 'Gutierrez', '😟.'  (token sucio),
#     mientras spaCy separa el punto y el emoji como tokens propios.
#  2) El split no separa 'https://chiapasparalelo.com/nota1' ni el punto final;
#     spaCy aisla la URL y la puntuacion, dejando tokens mas limpios para filtrar despues.

Ingenuo: ['Las', 'fuertes', 'lluvias', 'provocaron', 'inundaciones', 'en', 'varias', 'colonias', 'del', 'sur', 'de', 'Tuxtla', 'Gutierrez', '😟.']
spaCy  : ['Las', ' ', 'fuertes', 'lluvias', '  ', 'provocaron', 'inundaciones', 'en', 'varias', 'colonias', 'del', 'sur', 'de', 'Tuxtla', 'Gutierrez', '😟', '.', 'Proteccion']


**2.b** Normalización.

In [8]:
def _quitar_diacriticos(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

def normalizar(texto, quitar_acentos=True):
    """texto crudo -> string normalizado (aun sin tokenizar)."""
    t = texto.lower()
    t = unicodedata.normalize('NFC', t)
    t = RE_URL.sub(' ', t)
    t = RE_HTML.sub(' ', t)
    if quitar_acentos:
        t = _quitar_diacriticos(t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

print(normalizar(corpus_crudo[2]['texto']))

el cafe de chiapas rompio su record historico de exportacion este ciclo, impulsado por la demanda en europa y asia. los productores de la sierra celebran precios al alza.


In [15]:
print(corpus_crudo[2]['texto'])

<p>El <b>cafe</b> de Chiapas rompio su record historico de exportacion este ciclo, impulsado por la demanda en Europa y Asia.</p> Los productores de la Sierra celebran precios al alza.


**Respuesta modelo (2.b — acentos).** *A favor de quitarlos:* en un buscador el usuario casi nunca escribe acentos ('inundacion', 'cafe'), así que quitarlos sube el **recall** y colapsa 'inundación/inundacion' en un término. *En contra:* colapsa palabras distintas ('papá/papa', 'él/el'), perdiendo precisión y pistas de entidad. **Decisión por defecto: quitarlos** (`quitar_acentos=True`), porque para *retrieval* el recall pesa más que esos pocos pares ambiguos. (Para análisis lingüístico o NER la decisión se invertiría.)


## 3 · Stopwords con criterio

In [9]:
stopwords_es = nlp.Defaults.stop_words
print('Total stopwords spaCy (es):', len(stopwords_es))
print('¿estan estas?', {w: (w in stopwords_es) for w in ['no','nunca','sin','muy','poco']})

Total stopwords spaCy (es): 521
¿estan estas? {'no': True, 'nunca': True, 'sin': True, 'muy': True, 'poco': True}


In [10]:
# Conservamos negaciones e intensificadores: invierten o modulan el sentido.
CONSERVAR = {'no', 'nunca', 'sin', 'ni', 'muy', 'poco', 'nada', 'tampoco'}
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR
print(f'spaCy: {len(stopwords_es)}  ->  propias: {len(MIS_STOPWORDS)}  (conservadas {len(CONSERVAR & stopwords_es)})')
# verificacion de efecto sobre un caso critico
demo = 'no me gusto el servicio'
print('Con lista propia:', [w for w in normalizar(demo).split() if w not in MIS_STOPWORDS])

spaCy: 521  ->  propias: 513  (conservadas 8)
Con lista propia: ['no', 'gusto', 'servicio']


**Respuesta modelo (3).** Una respuesta sólida **conserva al menos las negaciones** ('no', 'nunca', 'sin', 'ni') y ofrece la razón: en sentiment/clasificación 'no me gustó' se volvería 'gustó servicio' (polaridad invertida) si 'no' se filtra. Para búsqueda pura el impacto es menor, pero conservarlas no daña y protege tareas downstream. Lo inaceptable es aplicar la lista de spaCy **sin haberla abierto**.


## 4 · Stemming vs. lemmatización

In [11]:
stemmer = SnowballStemmer('spanish')

def tokens_stemming(texto):
    base = normalizar(texto, quitar_acentos=True)
    toks = re.findall(r'[a-z0-9]+', base)
    return [stemmer.stem(w) for w in toks if w not in MIS_STOPWORDS and len(w) > 2]

def tokens_lemma(texto):
    # Para lematizar conviene NO quitar acentos antes (spaCy los espera);
    # se quitan despues, sobre el lema, para alinear con la decision del buscador.
    base = normalizar(texto, quitar_acentos=False)
    doc = nlp(base)
    out = []
    for tok in doc:
        if tok.is_punct or tok.is_space or tok.like_num or tok.is_stop:
            continue
        lema = _quitar_diacriticos(tok.lemma_.lower())
        if lema in MIS_STOPWORDS or len(lema) <= 2:
            continue
        out.append(lema)
    return out

print('stem :', tokens_stemming(corpus_crudo[3]['texto'])[:10])
print('lemma:', tokens_lemma(corpus_crudo[3]['texto'])[:10])

stem : ['sequi', 'afect', 'gravement', 'cultiv', 'maiz', 'frijol', 'region', 'fronteriz', 'agricultor', 'report']
lemma: ['sequia', 'afectar', 'gravemente', 'cultivo', 'maiz', 'frijol', 'region', 'fronterizo', 'agricultor', 'reportar']


In [12]:
V_stem  = {t for f in corpus_crudo for t in tokens_stemming(f['texto'])}
V_lemma = {t for f in corpus_crudo for t in tokens_lemma(f['texto'])}
print(f'|V_stemming| = {len(V_stem)}')
print(f'|V_lemma|    = {len(V_lemma)}')

# 10 ejemplos donde difieren: tomamos una palabra y vemos a que la manda cada metodo
muestra = ['corriendo','inundaciones','productores','celebran','restablecera','agricultores',
           'visitantes','exportacion','recorridos','cultivos']
print('\npalabra           stemming        lemma')
for w in muestra:
    print(f'{w:18}{stemmer.stem(w):16}{_quitar_diacriticos(nlp(w)[0].lemma_.lower())}')

|V_stemming| = 189
|V_lemma|    = 198

palabra           stemming        lemma
corriendo         corr            correr
inundaciones      inund           inundacion
productores       productor       productor
celebran          celebr          celebrar
restablecera      restablecer     restablecera
agricultores      agricultor      agricultor
visitantes        visit           visitante
exportacion       export          exportacion
recorridos        recorr          recorrido
cultivos          cultiv          cultivo


**Respuesta modelo (4 — decisión).** Se elige **lemmatización** para este corpus en español. El alumno debe **citar los números impresos**: típicamente el stemming produce un vocabulario algo menor pero con fragmentos no-palabra ('corr', 'celebr', 'inundac') que colapsan términos que deberían distinguirse, mientras la lematización devuelve formas de diccionario ('correr', 'celebrar', 'inundación') lingüísticamente válidas. Dada la riqueza morfológica del español (la conjugación verbal genera decenas de formas), el stemming heurístico de Snowball sobre-colapsa; la lematización con análisis morfológico es la opción correcta. *Penaliza* respuestas que deciden 'por intuición' sin apoyarse en los números ni en los ejemplos divergentes.


## 5 · Pipeline final + persistencia

In [13]:
def preprocesar(texto):
    """Pipeline definitivo: acentos quitados, negaciones conservadas, lematizacion."""
    return tokens_lemma(texto)

for fila in corpus_crudo[:3]:
    print(fila['id'], '->', preprocesar(fila['texto'])[:10])

d01 -> ['fuerte', 'lluvia', 'provocar', 'inundacion', 'colonia', 'sur', 'tuxtla', 'gutierrez', 'proteccion', 'civil']
d02 -> ['crisis', 'hidrico', 'agravar', 'desabasto', 'liquido', 'vital', 'afectar', 'mil', 'familia', 'zona']
d03 -> ['cafe', 'chiapa', 'rompio', 'record', 'historico', 'exportacion', 'ciclo', 'impulsado', 'demanda', 'europa']


In [14]:
corpus_procesado = [{'id': f['id'], 'titulo': f['titulo'],
                     'tokens': preprocesar(f['texto'])} for f in corpus_crudo]
with open('corpus_crudo.json', 'w', encoding='utf-8') as fh:
    json.dump(corpus_crudo, fh, ensure_ascii=False, indent=2)
with open('corpus_procesado.json', 'w', encoding='utf-8') as fh:
    json.dump(corpus_procesado, fh, ensure_ascii=False, indent=2)
print('Guardados corpus_crudo.json y corpus_procesado.json')

Guardados corpus_crudo.json y corpus_procesado.json


## Rúbrica sugerida (Lab 1)

| Criterio | Puntos |
|---|---|
| Exploración correcta (1.a, 1.b) | 15 |
| `normalizar` correcta + decisión de acentos justificada | 20 |
| Curación de stopwords con criterio de dominio | 15 |
| `tokens_stemming` y `tokens_lemma` correctos + comparación medida | 25 |
| `preprocesar` final coherente + JSON generado | 15 |
| Defensa oral (transversal, **eliminatoria**) | 10 |

_La defensa oral es eliminatoria: si el alumno no explica una de sus tres decisiones, la entrega se reprueba independientemente del puntaje numérico._
